In [13]:
import random
import time
from baselines import logger
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from yawning_titan.envs.generic.core.blue_interface import BlueInterface
from yawning_titan.envs.generic.core.red_interface import RedInterface
from yawning_titan.game_modes.game_mode import GameMode
from yawning_titan.envs.generic.generic_env import GenericNetworkEnv
from yawning_titan.envs.generic.core.network_interface import NetworkInterface
from yawning_titan.networks.network_db import default_18_node_network
from yawning_titan.game_modes.game_mode_db import default_game_mode
 
    
#TEST FOR INTEGRATING QLEARNING ALGORITHM FROM https://github.com/RodrigoToroIcarte/reward_machines/blob/master/reward_machines/rl_agents/qlearning/qlearning.py

# Define network and game mode
network = default_18_node_network()
#game_mode = default_game_mode()

game_mode = GameMode()
game_mode = game_mode.create_from_yaml(yaml='game_mode_balanced_revised.yaml', legacy=False, infer_legacy=True)


# Define interfaces
network_interface = NetworkInterface(game_mode=game_mode, network=network)
red = RedInterface(network_interface)
blue = BlueInterface(network_interface)

# Define environment
env = GenericNetworkEnv(red, blue, network_interface, print_metrics=True)
check_env(env, warn=True)

# Monitor environment
env = Monitor(env)

# Q-Learning based method
def get_qmax(Q, s, actions, q_init):
    if s not in Q:
        Q[s] = dict([(a, q_init) for a in actions])
    return max(Q[s].values())

def get_best_action(Q, s, actions, q_init):
    qmax = get_qmax(Q, s, actions, q_init)
    best = [a for a in actions if Q[s][a] == qmax]
    return random.choice(best)

def learn(env,
          network=None,
          seed=None,
          lr=0.1,
          total_timesteps=100000,
          epsilon=0.1,
          print_freq=10000,
          gamma=0.9,
          q_init=2.0,
          use_crm=False,
          use_rs=False):
    reward_total = 0
    step = 0
    num_episodes = 0
    Q = {}
    actions = list(range(env.action_space.n))

    while step < total_timesteps:
        s = tuple(env.reset())
        if s not in Q: Q[s] = dict([(a, q_init) for a in actions])
        while True:
            a = random.choice(actions) if random.random() < epsilon else get_best_action(Q, s, actions, q_init)
            sn, r, done, info = env.step(a)
            sn = tuple(sn)

            experiences = []
            if use_crm:
                for _s, _a, _r, _sn, _done in info["crm-experience"]:
                    experiences.append((tuple(_s), _a, _r, tuple(_sn), _done))
            elif use_rs:
                experiences = [(s, a, info["rs-reward"], sn, done)]
            else:
                experiences = [(s, a, r, sn, done)]

            for _s, _a, _r, _sn, _done in experiences:
                if _s not in Q: Q[_s] = dict([(b, q_init) for b in actions])
                if _done: _delta = _r - Q[_s][_a]
                else: _delta = _r + gamma * get_qmax(Q, _sn, actions, q_init) - Q[_s][_a]
                Q[_s][_a] += lr * _delta

            reward_total += r
            step += 1
            if step % print_freq == 0:
                logger.record_tabular("steps", step)
                logger.record_tabular("episodes", num_episodes)
                logger.record_tabular("total reward", reward_total)
                logger.dump_tabular()
                reward_total = 0
            if done:
                num_episodes += 1
                break
            s = sn

# Train the agent using Q-Learning
learn(env, lr=0.1, total_timesteps=2000, epsilon=0.1, print_freq=1000, gamma=0.9, q_init=2.0)


--Game over--
Total number of Games Played:  1
red wins!
Episode length:  835
Action          Avg Times Used  Percentage of Action Usage
------------  ----------------  ----------------------------
connect                    282  33.77%
restore_node               262  31.38%
isolate                    260  31.14%
do_nothing                  21  2.51%
scan                        10  1.2%



----------------------------
| episodes     | 1         |
| steps        | 1e+03     |
| total reward | -9.72e+03 |
----------------------------
--Game over--
Total number of Games Played:  2
blue wins!
Episode length:  1000
Action          Avg Times Used  Percentage of Action Usage
------------  ----------------  ----------------------------
isolate                    346  34.6%
connect                    317  31.7%
restore_node               304  30.4%
do_nothing                  17  1.7%
scan                        16  1.6%



----------------------------
| episodes     | 2         |
| steps      